In [1]:
import json
import datasets

In [2]:
yue_zh_characters_validation = datasets.load_dataset("google/smol", "gatitos__yue_zh")

In [3]:
with open("character_classification.json", "r") as f:
    character_classfication = json.load(f)

In [4]:
def canto_characters_in_sentence(sentence):
    canto_chars = []
    for x in sentence:
        if x in character_classfication.keys():
            if character_classfication[x] == "CANTO_ONLY":
                canto_chars.append(x)
    return canto_chars

In [5]:
with open("bad_translations.json", "r", encoding="utf-8") as f:
    bad_translations = json.load(f)

In [6]:
yue_zh_characters_validation = datasets.load_dataset("google/smol", "gatitos__yue_zh")

In [7]:
def replace_canto_only_characters(s: str) -> None | str:
    s_out = []
    for c in s:
        if c in character_classfication and character_classfication[c] == 'CANTO_ONLY' and c in yue_zh_characters_validation['train']['src']: # type: ignore
            token_translations = [x for x in yue_zh_characters_validation['train'] if x['src'] == c] # type: ignore
            s_out.append(token_translations[0]['trgs'][-1]) # type: ignore
        else:
            s_out.append(c)
            
    return ''.join(s_out)

In [10]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

characters = yue_zh_characters_validation["train"].to_pandas().sort_values(by="src", key=lambda x:x.str.len(), ascending=False)

def swap_tokens(x):
    z = ""
    i = 0
    while i < len(x):
        found = False
        for k in range(len(characters)):
            if characters["src"][k] == x[i:i+len(characters["src"][k] )]:
                z = z + characters["trgs"][k][0].split(";")[0]
                i += len(characters["src"][k])
                found = True
                break
        if not found:
            z += x[i]
            i += 1
    
    return z

model_name = 'jbochi/madlad400-3b-mt'
model = T5ForConditionalGeneration.from_pretrained(model_name, device_map=None)
tokenizer = T5Tokenizer.from_pretrained(model_name)

results = []

def translate(cantonese: str) -> str:
    text = f"<2en> {cantonese}"
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    outputs = model.generate(input_ids=input_ids, max_new_tokens=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

for item in bad_translations:
    translation_plain = translate(item['original_yue'])
    translation_swapped_tokens = translate(swap_tokens(item['original_yue']))
    translation_replaced_chars = translate(replace_canto_only_characters(item['original_yue'])) # type: ignore

    record = {
        "original": item['original'],
        "cantonese": translation_plain,
        "tkn-swaps": translation_swapped_tokens,
        "chr-swaps": translation_replaced_chars,
    }

    results.append(record)

with open('sentence_translations_madlad_token_replacements.json', 'w+') as f:
    json.dump(results, f, indent=4)


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 237bd03d-7482-48e0-bfab-40a1e3442836)')' thrown while requesting HEAD https://huggingface.co/jbochi/madlad400-3b-mt/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
